In [ ]:
###################### Hybrid Retrieval-Augmented Generation (RAG) with Multi-Query and Reranking ######################

In [ ]:
# Hybrid RAG with Chroma DB Cloud as Hybrid search can be done only with Cloud Chroma DB

In [105]:
# Load the documents

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('data/subtitles', glob="*.srt", show_progress=True, loader_cls=TextLoader)
kb_docs = loader.load()

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 118.94it/s]


In [106]:
# Chunk the loaded documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(kb_docs)

In [107]:
# Initialize Chroma Client + Schema

import chromadb
from chromadb import Schema, SparseVectorIndexConfig, VectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction, OpenAIEmbeddingFunction
from chromadb.utils import embedding_functions

# Load API Keys
f = open('keys/.chroma_api_key.txt')
CHROMA_API_KEY = f.read()

f = open('keys/.hugging_api_key.txt')
HUGGING_API_KEY = f.read()

# Initialize Chroma Cloud Client

client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant='27ff1a2e-cec2-46d9-8db6-3e4a51f30782',
  database='chroma_vector_db'
)

# Create Hybrid Schema : Sparse Index + Dense Index
# Sparse Index
schema = Schema()
    
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

# Dense Index

# Configure vector index with custom embedding function
hf_ef = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=HUGGING_API_KEY,
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
        embedding_function=hf_ef
    )
)

# Create Collection

collection = client.get_or_create_collection(
  name="friends_collection",
  schema=schema,
)

In [108]:
# Insert Documents

import uuid

# Free tier in chroma supports 300 docs to be inserted at a time
BATCH_SIZE = 250

for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]

    collection.add(
        documents=[chunk.page_content for chunk in batch],
        metadatas=[chunk.metadata for chunk in batch],
        ids=[str(uuid.uuid4()) for _ in batch]
    )

    print(f"Inserted {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)}")

Inserted 250/514
Inserted 500/514
Inserted 514/514


In [110]:
# Hybrid Retriever

from chromadb import Knn, K
from chromadb import Search
from chromadb import Rrf

def hybrid_retriever(query):
    # Sparse embeddings
    sparse_rank = Knn(
      query=query,  # Text query for sparse embeddings
      key="sparse_vector_key",  # Metadata field for sparse vectors
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)
    )
    
    # Dense semantic embeddings
    dense_rank = Knn(
      query=query,  # Text query for dense embeddings
      key="#embedding",          # Default embedding field
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)

    )
    
    # Combine with RRF (Reciprocal Rank Fusion). RRF is a ranking algorithm that combines multiple ranked lists using a formula
    hybrid_rank = Rrf(
      ranks=[dense_rank, sparse_rank],
      weights=[0.7, 0.3],  # 70% semantic, 30% keyword
      k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
    )
    
    # Use in search
    hybrid_search = (Search()
      .rank(hybrid_rank)
      .limit(5)
      .select(K.DOCUMENT, K.SCORE, K.METADATA)
    )
    
    hybrid_results = collection.search(hybrid_search)
    
    return hybrid_results

In [111]:
#  Initialize LLM

from langchain_google_genai import ChatGoogleGenerativeAI

f = open("keys/.google_api_key.txt")
GOOGLE_API_KEY = f.read()

# Initialize LLM
chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

In [112]:
# Multi-Query Generator : Retrieve Multi-Queries for the user prompt

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MULTI_QUERY_PROMPT = """
You are an AI assistant that generates multiple search queries 
to retrieve documents from a vector database.

Generate 4 different versions of the following question.
Each query should capture a different way the question might be asked.

Original Question:
{question}

Return each query on a new line.
"""

multi_query_prompt = ChatPromptTemplate.from_template(MULTI_QUERY_PROMPT)

# chain
multi_query_chain = multi_query_prompt | chat_model | StrOutputParser()

In [113]:
# for testing
response = multi_query_chain.invoke({"question": "who is Mahesh"})
print(response)

What information is available about Mahesh?
Can you tell me about Mahesh?
Who exactly is Mahesh?
Details about Mahesh


In [114]:
def generate_queries(question):
    response = multi_query_chain.invoke({"question": question})

    queries = [q.strip() for q in response.split("\n") if q.strip()]
    
    # include original query
    queries.append(question)

    return list(set(queries))

In [115]:
print(generate_queries("who is Mahesh"))

['What information is available about Mahesh?', 'Who is Mahesh and what is his significance?', 'who is Mahesh', 'Can you tell me about Mahesh?', "Details regarding Mahesh's identity or role."]


In [116]:
# Multi Query Retrieval
def multi_query_hybrid_retriever(query):

    # Use LLM to get multiple queries for the given user prompt
    queries = generate_queries(query)

    all_docs = {}
    
    for q in queries:
        # send each query to Hybrid retriever
        results = hybrid_retriever(q)

        for row in results.rows()[0]:
            doc = row["document"]
            score = row["score"]

            # deduplicate using document text
            if doc not in all_docs:
                all_docs[doc] = score
            else:
                # keep best score
                all_docs[doc] = max(all_docs[doc], score)

    # sort by score
    sorted_docs = sorted(all_docs.items(), key=lambda x: x[1], reverse=True)

    # keep top 5
    final_docs = [{"document": doc, "score": score} for doc, score in sorted_docs[:5]]

    return final_docs

In [117]:
print(multi_query_hybrid_retriever("who is Mahesh"))

[{'document': "169\n00:10:46,982 --> 00:10:50,577\n...who locked up your mom\nand stole her Gremlin.\n\n170\n00:10:52,020 --> 00:10:56,252\nI know. I just wanted to know\nwho he was, you know?\n\n171\n00:10:58,326 --> 00:10:59,759\nYeah, I know.\n\n172\n00:11:02,531 --> 00:11:06,558\nI wasn't completely honest with you\nwhen I said I didn't know...\n\n173\n00:11:06,768 --> 00:11:08,998\n...exactly where he lived.\n\n174\n00:11:09,337 --> 00:11:10,964\nWhat do you mean?", 'score': -0.0139296185}]


In [92]:
# for Reranking

from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [119]:
# for Reranking
def rerank_documents(query, docs, top_k=5):

    #pairs = [[query, doc.page_content] for doc in docs]
    pairs = [[query, doc["document"]] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = list(zip(docs, scores))

    ranked_docs = sorted(
        scored_docs,
        key=lambda x: x[1],
        reverse=True
    )

    return [doc for doc, score in ranked_docs[:top_k]]

# Retrieve + Rerank
def retrieve_and_rerank(query):
    docs = multi_query_hybrid_retriever(query)
    reranked_docs = rerank_documents(query, docs)

    return reranked_docs

In [120]:
# Format Context
def format_docs(docs):

    context = ""

    for row in docs:
        context += row["document"] + "\n\n"

    return context

In [121]:
# Initialize a Chat Prompt Template

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

# Initialize a Output Parser
parser = StrOutputParser()

generator_chain = prompt_template | chat_model | parser

In [122]:
# Runnable Wrappers

from langchain_core.runnables import RunnableLambda

# multi_query_retriever_runnable = RunnableLambda(
#     lambda x: multi_query_hybrid_retriever(x)
# )

format_docs_runnable = RunnableLambda(lambda x: format_docs(x))

# make it runnable
retrieve_and_rerank_runnable = RunnableLambda(retrieve_and_rerank)

In [123]:
# Final RAG Chain

from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retrieve_and_rerank_runnable | format_docs_runnable,
    "question": RunnablePassthrough()
} | generator_chain

In [125]:
query = "Who is Rachel?"

rag_chain.invoke(query)

"Rachel is someone Ross has been in love with since forever. Ross tried to tell her his feelings, but something always got in the way. Chandler let it slip that Ross loved Rachel. Monica calls to ask if Rachel is working. Rachel has friends. She might have gone to Bloomingdale's."